In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely
from shapely.affinity import translate, rotate   # ← add rotate
import os

# ---------------------------------------------------------------------
# File paths (adjust if needed)
# ---------------------------------------------------------------------
grid_path = ""
treatment_path = ""
padus_path = ""
mtbs_path = ""
urban_path = ""
roads_path = ""

output_dir = ""
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------------------------
# Load Data
# ---------------------------------------------------------------------
print("Loading data...")
grid = gpd.read_file(grid_path)
treatment_sites = gpd.read_file(treatment_path)
padus = gpd.read_file(padus_path)
mtbs = gpd.read_file(mtbs_path)
urbandata = gpd.read_file(urban_path)
roads = gpd.read_file(roads_path)

# Ensure consistent CRS
crs = grid.crs
for df in [padus, mtbs, urbandata, roads, treatment_sites]:
    df.to_crs(crs, inplace=True)

# Clip PADUS + MTBS to the bounding box of the grid (for speed)
grid_bounds = grid.total_bounds
padus = padus.cx[grid_bounds[0]:grid_bounds[2], grid_bounds[1]:grid_bounds[3]].copy()
mtbs = mtbs.cx[grid_bounds[0]:grid_bounds[2], grid_bounds[1]:grid_bounds[3]].copy()

# Filter MTBS (2010+)
mtbs["ig_year"] = pd.to_datetime(mtbs["Ig_Date"], errors="coerce").dt.year
mtbs_recent = mtbs[mtbs["ig_year"] >= 2010].copy()

# Precompute flags on grid
print("Precomputing public land and MTBS overlap flags...")
grid["public_overlap"] = grid.geometry.apply(lambda x: padus.intersects(x).any())
grid["burned_since_2010"] = grid.geometry.apply(lambda x: mtbs_recent.intersects(x).any())
grid["touches_public"] = grid.centroid.apply(lambda x: padus.intersects(x).any())

# The features used for matching
match_features = ["Access_U_1", "trail_leng", "proxy_coun", "Elevation", "Slope", "Temperatur", "Precipitat"]

# ---------------------------------------------------------------------
# Determine remaining treatment sites to process
# ---------------------------------------------------------------------
treatment_ids = treatment_sites["Incid_Name"].unique()
existing_controls = [
    fname.replace("control_raw_", "").replace(".shp", "")
    for fname in os.listdir(output_dir)
    if fname.startswith("control_raw_") and fname.endswith(".shp")
]
remaining_ids = [fid for fid in treatment_ids if fid not in existing_controls]

print(f"\n⏭ Skipping {len(treatment_ids) - len(remaining_ids)} already matched treatment sites.")
print(f"▶️ Proceeding with {len(remaining_ids)} remaining treatments:")
print(", ".join(remaining_ids))

# ---------------------------------------------------------------------
# Matching Function (now with rotation sweep at each seed)
# ---------------------------------------------------------------------
def process_treatment(fire_name):
    try:
        out_shp_raw = os.path.join(output_dir, f"control_raw_{fire_name}.shp")
        out_shp_final = os.path.join(output_dir, f"control_final_{fire_name}.shp")
        out_csv = os.path.join(output_dir, f"comparison_{fire_name}.csv")

        selected_treatment = treatment_sites[treatment_sites["Incid_Name"] == fire_name]
        if selected_treatment.empty:
            return f"Skipping {fire_name} (no treatment found)"

        selected_geom = selected_treatment.unary_union
        treatment_area = selected_geom.area
        area_min, area_max = 0.5 * treatment_area, 1.5 * treatment_area

        # Build treatment vector (sum for length/proxy, mean for others)
        treat_stats = {
            f: selected_treatment[f].sum() if f in ["trail_leng", "proxy_coun"] else selected_treatment[f].mean()
            for f in match_features
        }
        treatment_vector = np.array([[treat_stats[f] for f in match_features]], dtype=float)

        best_score = np.inf
        best_site = None
        best_cluster_stats = None

        treat_centroid = selected_geom.centroid
        candidate_seeds = grid[grid["touches_public"]]

        for _, seed_cell in candidate_seeds.iterrows():
            # keep ≥ 80,467.2 m (~50 miles) away
            if seed_cell.geometry.centroid.distance(treat_centroid) < 80467.2:
                continue

            dx = seed_cell.geometry.centroid.x - treat_centroid.x
            dy = seed_cell.geometry.centroid.y - treat_centroid.y

            # ---- rotation sweep (like in your propensity script) ----
            for ang in range(0, 360, 45):
                rotated = rotate(selected_geom, ang, origin="center")
                shifted_geom = translate(rotated, xoff=dx, yoff=dy)

                window_cells = grid[
                    grid.geometry.intersects(shifted_geom)
                    & grid["public_overlap"]
                    & ~grid["burned_since_2010"]
                ]
                if window_cells.empty:
                    continue

                window_area = window_cells.geometry.area.sum()
                if not (area_min <= window_area <= area_max):
                    continue

                cluster_stats = {
                    f: window_cells[f].sum() if f in ["trail_leng", "proxy_coun"] else window_cells[f].mean()
                    for f in match_features
                }
                cluster_vector = np.array([[cluster_stats[f] for f in match_features]], dtype=float)
                dist = np.linalg.norm(cluster_vector - treatment_vector)

                if dist < best_score:
                    best_score = dist
                    best_site = window_cells.copy()
                    best_cluster_stats = cluster_stats

        if best_site is None or best_site.empty:
            print(f"No valid control for {fire_name}")
            return f"No valid control for {fire_name}"

        print(f"Matched control for {fire_name} with distance = {best_score:.3f}")

        # Export RAW cluster cells
        raw_gdf = best_site.copy()
        raw_gdf["geometry"] = raw_gdf["geometry"].buffer(0)
        raw_gdf.to_file(out_shp_raw)
        print(f"Exported RAW cluster shapefile: {out_shp_raw}")

        # Build final control geometry: clip to PADUS, remove urban + 100m buffered roads
        control_union = raw_gdf.unary_union
        padus_union = padus.unary_union
        after_padus = control_union.intersection(padus_union)

        roads_union = roads.buffer(100).unary_union
        urban_union = urbandata.unary_union
        roads_urban_union = urban_union.union(roads_union)

        final_geom = after_padus.difference(roads_urban_union)
        if final_geom.is_empty:
            print(f"final_geom is empty. No final shapefile for {fire_name}.")
        else:
            final_gdf = gpd.GeoDataFrame(
                raw_gdf.iloc[:1].copy(),
                geometry=[final_geom],
                crs=grid.crs
            )
            final_gdf.to_file(out_shp_final)
            print(f"Exported FINAL geometry shapefile: {out_shp_final}")

        # Export comparison CSV
        comparison_df = pd.DataFrame({
            "Metric": match_features,
            "Treatment": [treat_stats[f] for f in match_features],
            "Control": [best_cluster_stats[f] for f in match_features],
            "Abs_Diff": [abs(treat_stats[f] - best_cluster_stats[f]) for f in match_features]
        })
        comparison_df.to_csv(out_csv, index=False)
        print(f"Exported comparison CSV: {out_csv}")

        return f"Completed {fire_name}"

    except Exception as e:
        return f"Error processing {fire_name}: {e}"

# ---------------------------------------------------------------------
# Process Remaining Treatments
# ---------------------------------------------------------------------
results = []
print(f"\n=== Resuming with {len(remaining_ids)} treatment sites ===")
for i, tid in enumerate(remaining_ids, 1):
    print(f"\n[{i}/{len(remaining_ids)}] Processing {tid}...")
    results.append(process_treatment(tid))

# Summary
print("\n=== Summary ===")
for r in results:
    print(r)

# ---------------------------------------------------------------------
# Merge all exported *final* control shapefiles
# ---------------------------------------------------------------------
print("\nMerging all final control shapefiles into one (if any exist)...")
all_controls = []
for fname in os.listdir(output_dir):
    if fname.startswith("control_final_") and fname.endswith(".shp"):
        shp_path = os.path.join(output_dir, fname)
        try:
            gdf = gpd.read_file(shp_path)
            all_controls.append(gdf)
        except Exception as e:
            print(f"Warning: Failed to load {fname}: {e}")

if all_controls:
    merged_controls = pd.concat(all_controls, ignore_index=True)
    merged_shp = os.path.join(output_dir, "all_controls_merged.shp")
    merged_controls.to_file(merged_shp)
    print(f"Merged final control shapefile exported: {merged_shp}")
else:
    print("No final shapefiles found to merge.")
